# Quantum Computing Based Security of Multimedia Traffic in 6G Networks — experiments
Firas Ftoutou — M2R SIIVA, ISI

This notebook reproduces all the results of the paper. No external dataset is used: all data (keys, messages, the test video, BB84 photons) is generated by the code.
1. Cost of one key exchange (classical vs post-quantum vs hybrid)
2. Cost of one signature
3. Encryption of a real H.264 video with AES-256
4. Extra traffic of the key-change rule
5. BB84 quantum key distribution simulation

**Run it in Google Colab: Runtime → Run all.** The first cell installs the libraries (about 3–5 minutes).

In [ ]:
# 0. Install the libraries
# - cryptography : classical algorithms (X25519, RSA, Ed25519, AES)
# - liboqs-python: NIST post-quantum algorithms (ML-KEM, ML-DSA), from the Open Quantum Safe project
!pip -q install cryptography liboqs-python
import oqs   # the first import downloads and builds liboqs (a few minutes)
print("liboqs version:", oqs.oqs_version())

In [ ]:
import os, time, statistics
import numpy as np
from cryptography.hazmat.primitives.asymmetric import x25519, rsa, padding, ed25519
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

def median_time_us(function, runs=1000):
    """Run a function many times and return the median time in microseconds."""
    function()                                   # warm-up run
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        function()
        times.append((time.perf_counter() - start) * 1e6)
    return statistics.median(times)

## 1. Cost of one key exchange
One full exchange = both sides (client and server) compute the shared secret.

In [ ]:
# Classical: X25519 (elliptic curve)
def x25519_exchange():
    client = x25519.X25519PrivateKey.generate()
    server = x25519.X25519PrivateKey.generate()
    secret_client = client.exchange(server.public_key())
    secret_server = server.exchange(client.public_key())
    assert secret_client == secret_server
    return secret_client

# Classical: RSA-3072 (the server already has its key pair)
rsa_key = rsa.generate_private_key(public_exponent=65537, key_size=3072)
oaep = padding.OAEP(mgf=padding.MGF1(hashes.SHA256()), algorithm=hashes.SHA256(), label=None)
def rsa_exchange():
    secret = os.urandom(32)                               # client chooses a secret
    ciphertext = rsa_key.public_key().encrypt(secret, oaep)   # and encrypts it for the server
    return rsa_key.decrypt(ciphertext, oaep)              # server decrypts it

# Post-quantum: ML-KEM-768 (NIST FIPS 203)
def mlkem_exchange():
    with oqs.KeyEncapsulation("ML-KEM-768") as client, oqs.KeyEncapsulation("ML-KEM-768") as server:
        public_key = client.generate_keypair()                    # client sends its public key
        ciphertext, secret_server = server.encap_secret(public_key)  # server sends a ciphertext
        secret_client = client.decap_secret(ciphertext)           # client recovers the secret
        assert secret_client == secret_server
        return secret_client

# Hybrid: X25519 + ML-KEM-768, both secrets mixed into one key
def hybrid_exchange():
    s1 = x25519_exchange()
    s2 = mlkem_exchange()
    return HKDF(algorithm=hashes.SHA256(), length=32, salt=None, info=b"hybrid").derive(s1 + s2)

mlkem = oqs.KeyEncapsulation("ML-KEM-768").details
mlkem_bytes = mlkem["length_public_key"] + mlkem["length_ciphertext"]

results = {
    "X25519":     (median_time_us(x25519_exchange), 32 + 32),
    "RSA-3072":   (median_time_us(rsa_exchange, 300), 384 + 384),
    "ML-KEM-768": (median_time_us(mlkem_exchange), mlkem_bytes),
    "Hybrid":     (median_time_us(hybrid_exchange), 64 + mlkem_bytes),
}
print(f"{'Algorithm':<12}{'Time (us)':>12}{'Bytes sent':>12}")
for name, (t, b) in results.items():
    print(f"{name:<12}{t:>12.1f}{b:>12}")

## 2. Cost of one signature (sign + verify)

In [ ]:
message = os.urandom(256)

ed_key = ed25519.Ed25519PrivateKey.generate()
def ed25519_sign_verify():
    signature = ed_key.sign(message)
    ed_key.public_key().verify(signature, message)

mldsa = oqs.Signature("ML-DSA-44")
mldsa_public = mldsa.generate_keypair()
def mldsa_sign_verify():
    signature = mldsa.sign(message)
    assert mldsa.verify(message, signature, mldsa_public)

sig_bytes = mldsa.details["length_signature"]
print(f"Ed25519   : {median_time_us(ed25519_sign_verify):6.1f} us, signature = 64 bytes")
print(f"ML-DSA-44 : {median_time_us(mldsa_sign_verify):6.1f} us, signature = {sig_bytes} bytes")

## 3. Encrypting a real H.264 video with AES-256-GCM
We create a 20-second Full HD video at 8 Mb/s, cut it into 1200-byte packets (like live streaming) and encrypt each packet.

In [ ]:
# Create a real H.264 video with ffmpeg (1080p, 30 images/s, 8 Mb/s, 20 s)
!ffmpeg -loglevel error -y -f lavfi -i testsrc2=size=1920x1080:rate=30 -t 20 -c:v libx264 -preset veryfast -b:v 8M -maxrate 8M -bufsize 16M -g 30 video.mp4

data = open("video.mp4", "rb").read()
packets = [data[i:i + 1200] for i in range(0, len(data), 1200)]
aes = AESGCM(os.urandom(32))                 # AES with a 256-bit key

start = time.perf_counter()
for number, packet in enumerate(packets):
    nonce = number.to_bytes(12, "big")       # a different nonce for every packet
    aes.encrypt(nonce, packet, None)
seconds = time.perf_counter() - start

print(f"Video size: {len(data)/1e6:.1f} MB, {len(packets)} packets")
print(f"AES-256-GCM speed: {len(data)*8/seconds/1e9:.1f} Gb/s (the video needs only 0.008 Gb/s)")
print(f"Extra bytes per packet: 28 (12 nonce + 16 tag) = {28/1200*100:.2f} %")

## 4. Key-change rule: extra traffic
One post-quantum key change = hybrid exchange + ML-DSA-44 signature.
Extra traffic = 8·B / (R·T). Rule: T ≥ 8·B / (0.01·R) keeps it below 1 %.

In [ ]:
B = 64 + mlkem_bytes + sig_bytes          # bytes of one key change
print("One key change sends B =", B, "bytes\n")

streams = {"HD video": 8e6, "4K video": 25e6, "XR": 100e6, "Hologram": 1e9}   # bit/s
print(f"{'Stream':<10}{'every 1 s':>11}{'every 0.1 s':>13}{'every 0.01 s':>14}{'min T (1 %)':>14}")
for name, R in streams.items():
    overhead = [8 * B / (R * T) * 100 for T in (1, 0.1, 0.01)]
    T_min = 8 * B / (0.01 * R)
    print(f"{name:<10}{overhead[0]:>10.3f}%{overhead[1]:>12.2f}%{overhead[2]:>13.1f}%{T_min:>12.3f} s")

## 5. BB84 simulation
Alice sends random bits in random bases; Bob measures in random bases; they keep only the bits where the bases match. A spy who measures and resends the photons creates about 25 % errors.

In [ ]:
rng = np.random.default_rng(0)

def bb84(n_photons, spy=False, noise=0.0):
    alice_bits  = rng.integers(0, 2, n_photons)
    alice_bases = rng.integers(0, 2, n_photons)
    bob_bases   = rng.integers(0, 2, n_photons)
    bits, bases = alice_bits.copy(), alice_bases.copy()
    if spy:                                    # the spy measures in a random basis and resends
        spy_bases = rng.integers(0, 2, n_photons)
        bits = np.where(spy_bases == bases, bits, rng.integers(0, 2, n_photons))
        bases = spy_bases
    # Bob gets the right bit if he uses the same basis, otherwise a random bit
    bob_bits = np.where(bob_bases == bases, bits, rng.integers(0, 2, n_photons))
    bob_bits = bob_bits ^ (rng.random(n_photons) < noise)   # channel noise
    keep = alice_bases == bob_bases                          # sifting: same basis only
    error_rate = np.mean(alice_bits[keep] != bob_bits[keep])
    return keep.mean(), error_rate

for case, spy, noise in [("No spy, 1 % noise", False, 0.01),
                         ("No spy, 5 % noise", False, 0.05),
                         ("Spy (measure and resend)", True, 0.01)]:
    kept, err = bb84(200_000, spy, noise)
    print(f"{case:<28} bits kept = {kept*100:4.1f} %   error rate = {err*100:4.1f} %")